# 05 — Full Reranking Pipeline Walkthrough

End-to-end: embedding candidates → Wikidata grounding → LLM reranking → metrics.

**Requires API key.**

In [ ]:
import sys; sys.path.insert(0, '..')
import random
import pandas as pd
import matplotlib.pyplot as plt
from src.config import Settings
from src.data.fb15k237 import FB15k237Dataset
from src.models.embedding_baseline import RotatEBaseline
from src.models.llm_client import LLMClient
from src.models.reranker import LLMReranker
from src.eval.candidates import generate_candidates
from src.eval.metrics import compute_metrics
from src.wikidata.sparql import WikidataGrounder
from src.wikidata.cache import SPARQLCache
from src.utils.cost_tracker import CostTracker
from src.utils.reproducibility import set_seed

settings = Settings()
set_seed(settings.random_seed)
assert settings.ai_gateway_api_key, 'Set AI_GATEWAY_API_KEY in .env!'

## 1. Load All Components

In [ ]:
dataset = FB15k237Dataset(data_dir=settings.data_dir)
dataset.load()
embed_model = RotatEBaseline(cache_dir=settings.cache_dir)
embed_model.load()
cache    = SPARQLCache(cache_dir=settings.cache_dir)
grounder = WikidataGrounder(
    sparql_url=settings.wikidata_sparql_url,
    user_agent=settings.wikidata_user_agent,
    cache=cache,
)
tracker  = CostTracker()
client   = LLMClient(settings=settings)
reranker = LLMReranker(
    client=client, grounder=grounder,
    cost_tracker=tracker, template='minimal',
)
print('All components loaded.')

## 2. Single Query Walkthrough

In [ ]:
h, r, t_true = dataset.test[0]
print(f'Query : ({h}, {r}, ?)')
print(f'Answer: {t_true}\n')
cands = generate_candidates(
    model=embed_model, head=h, relation=r,
    dataset=dataset, top_k=settings.num_candidates,
)
embed_rank = next((i+1 for i,(e,_) in enumerate(cands) if e==t_true), None)
print(f'Embedding rank: {embed_rank}')
reranked = reranker.rerank(head=h, relation=r, candidates=[e for e,_ in cands])
llm_rank = next((i+1 for i,(e,_) in enumerate(reranked) if e==t_true), None)
print(f'LLM rank      : {llm_rank}')
print(f'Cost so far   : ${tracker.total_cost:.5f}')

## 3. Evaluate Embedding vs LLM over N Queries

In [ ]:
random.seed(settings.random_seed)
queries = random.sample(dataset.test, settings.sample_test_queries)
embed_ranks, llm_ranks = [], []
for h,r,t_true in queries:
    cands = generate_candidates(model=embed_model,head=h,relation=r,
                                dataset=dataset,top_k=settings.num_candidates)
    er = next((i+1 for i,(e,_) in enumerate(cands) if e==t_true), len(cands)+1)
    embed_ranks.append(er)
    rr = reranker.rerank(head=h,relation=r,candidates=[e for e,_ in cands])
    lr = next((i+1 for i,(e,_) in enumerate(rr) if e==t_true), len(rr)+1)
    llm_ranks.append(lr)
em = compute_metrics(embed_ranks)
lm = compute_metrics(llm_ranks)
display(pd.DataFrame({'Embedding':em,'LLM Reranked':lm}))
print(f'Total cost: ${tracker.total_cost:.4f} ({tracker.total_calls} calls)')

## 4. Rank Distribution

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,4))
for ax,ranks,title in zip(axes,[embed_ranks,llm_ranks],['Embedding','LLM Reranked']):
    ax.hist(ranks,bins=20,color='steelblue',edgecolor='white')
    ax.set(title=f'{title} Rank Distribution',xlabel='Rank',ylabel='Count')
plt.tight_layout(); plt.show()

## 5. Per-Template Comparison

In [ ]:
TEMPLATES=['minimal','with_descriptions','strict_rubric','concise_cot','binary_judge']
results={}
for tname in TEMPLATES:
    tt=CostTracker()
    rr=LLMReranker(client=client,grounder=grounder,cost_tracker=tt,template=tname)
    ranks=[]
    for h,r,t_true in queries:
        cands=generate_candidates(model=embed_model,head=h,relation=r,
                                  dataset=dataset,top_k=settings.num_candidates)
        reranked=rr.rerank(head=h,relation=r,candidates=[e for e,_ in cands])
        rank=next((i+1 for i,(e,_) in enumerate(reranked) if e==t_true),len(reranked)+1)
        ranks.append(rank)
    m=compute_metrics(ranks); m['cost']=tt.total_cost; results[tname]=m
    print(f'{tname:<25} MRR={m["mrr"]:.4f}  cost=${m["cost"]:.4f}')
display(pd.DataFrame(results).T)